In [1]:
# %% [markdown]
# #  Simulation d'Entretien avec Groq (Llama 3.3 70B)
# 
# **Objectif** : Tester la génération de questions et l'évaluation des réponses
# 
# **Pipeline** :
# 1. Charger un CV et une offre
# 2. Générer des questions personnalisées
# 3. Simuler des réponses
# 4. Évaluer les réponses
# 5. Générer feedback global

In [2]:
import sys
sys.path.append('..')

import json
import os
from pathlib import Path
from src.interview_simulator import InterviewSimulator
from src.skills_extractor import SkillsExtractor
from src.cv_parser import CVParser

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError(
        " GROQ_API_KEY non trouvée !\n"
        "1. Créez un fichier .env à la racine\n"
        "2. Ajoutez : GROQ_API_KEY=votre_clé\n"
        "3. Obtenez une clé sur : https://console.groq.com/"
    )

print(f" Clé API chargée : {os.getenv('GROQ_API_KEY')[:20]}...")


print("="*60)
print(" INITIALISATION")
print("="*60)

try:
    simulator = InterviewSimulator()
    skills_extractor = SkillsExtractor()
    print(" Tous les modules chargés")
except ValueError as e:
    print(f" {e}")
    print("\n Obtenez une clé gratuite sur : https://console.groq.com/")
    raise

 Clé API chargée : gsk_dQ0Dt8JrxyLkGtT2...
 INITIALISATION
 InterviewSimulator initialisé avec llama-3.3-70b-versatile


INFO:src.skills_extractor:Modèle spaCy chargé : en_core_web_sm (Anglais (développement local))
INFO:src.skills_extractor: Base de compétences chargée depuis skills_reference.json
INFO:src.skills_extractor:   • Compétences techniques : 653
INFO:src.skills_extractor:   • Soft skills : 190


   • Variations : 472 mappings
 Tous les modules chargés


In [4]:
print("\n" + "="*60)
print(" CHARGEMENT DES DONNÉES")
print("="*60)

cv_path = Path("../data/RESUME Robert UNG.pdf")

if not cv_path.exists():
    print(f"  CV introuvable : {cv_path}")
    print("Utilisez votre propre CV pour tester")
    cv_skills = ["Python", "Machine Learning", "TensorFlow", "Docker", "FastAPI", "Git"]
else:
    parser = CVParser()
    cv_text = parser.parse(str(cv_path))
    skills_result = skills_extractor.extract_from_cv(cv_text)
    cv_skills = skills_result['technical_skills']

print(f"\n Compétences CV : {', '.join(cv_skills[:10])}")

with open('../data/jobs/jobs_dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
    job = dataset['jobs'][0]  

print(f"\n Offre sélectionnée")
print(f"   Titre : {job['title']}")
print(f"   Entreprise : {job['company']}")
print(f"   Requirements : {', '.join(job['requirements'][:5])}...")


 CHARGEMENT DES DONNÉES
  CV introuvable : ../data/RESUME Robert UNG.pdf
Utilisez votre propre CV pour tester

 Compétences CV : Python, Machine Learning, TensorFlow, Docker, FastAPI, Git

 Offre sélectionnée
   Titre : Junior ML Engineer
   Entreprise : AI Startup Paris
   Requirements : Python (numpy, pandas, scikit-learn), Machine Learning basics (supervised learning), Git et GitHub, Docker (notions de base), Anglais technique (lecture documentation)...


In [5]:
print("\n" + "="*60)
print(" GÉNÉRATION DES QUESTIONS")
print("="*60)

try:
    questions = simulator.generate_questions(
        cv_skills=cv_skills,
        job_title=job['title'],
        job_description=job['description'],
        job_requirements=job['requirements'],
        num_questions=8
    )
    
    print(f"\n {len(questions['rh_questions'])} questions RH générées")
    print(f" {len(questions['technical_questions'])} questions techniques générées")
    
except Exception as e:
    print(f"\n ERREUR lors de la génération des questions:")
    print(f"   {type(e).__name__}: {e}")
    print("\n Causes possibles:")
    print("   • Rate limiting Groq (429)")
    print("   • Clé API invalide")
    print("   • Problème de connexion")
    raise

print(f"\n QUESTIONS RH ({len(questions['rh_questions'])}):")
for q in questions['rh_questions']:
    print(f"\n{q['id']}. [{q['type']}]")
    print(f"   {q['question']}")

print(f"\n QUESTIONS TECHNIQUES ({len(questions['technical_questions'])}):")
for q in questions['technical_questions']:
    skill = q.get('skill', 'N/A')
    print(f"\n{q['id']}. [{q['type']} - {skill}]")
    print(f"   {q['question']}")


 GÉNÉRATION DES QUESTIONS


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 4 questions RH générées
 4 questions techniques générées

 4 questions RH générées
 4 questions techniques générées

 QUESTIONS RH (4):

1. [motivation]
   Qu'est-ce qui vous motive à rejoindre notre équipe en tant que Junior ML Engineer ?

2. [formation]
   Parlez-moi de votre parcours académique et comment il vous a préparé pour ce poste.

3. [adaptabilité]
   Décrivez une situation où vous avez dû apprendre rapidement une nouvelle technologie ou compétence.

4. [développement]
   Comment vous voyez-vous évoluer dans notre équipe et quels sont vos objectifs à court et moyen terme ?

 QUESTIONS TECHNIQUES (4):

5. [compétence_technique - Python]
   Comment utilisez-vous les bibliothèques Python comme NumPy et Pandas pour l'analyse de données ?

6. [concept - Machine Learning]
   Expliquez les principes de base du Machine Learning et comment vous les avez appliqués dans un projet.

7. [outil - Docker]
   Décrivez votre expérience avec Docker et comment vous l'utilisez pour déployer de

In [6]:

print("\n" + "="*60)
print(" SIMULATION DE RÉPONSES")
print("="*60)

simulated_answers = {
    1: """Je suis diplômé d'un Master en IA et j'ai développé plusieurs projets en Machine Learning pendant mes stages. 
    J'ai notamment travaillé sur un système de recommandation avec TensorFlow qui a amélioré l'engagement de 15%. 
    Je suis passionné par l'apprentissage automatique et cherche à approfondir mes compétences en production de modèles ML.""",
    
    2: """Je souhaite rejoindre votre entreprise car vous êtes leaders dans l'IA. J'ai suivi vos publications techniques 
    et je suis impressionné par vos innovations. Cette position me permettrait d'apprendre auprès d'experts tout en 
    contribuant à des projets impactants.""",
    
    5: """En Python, j'utilise pandas pour la manipulation de données. Pour nettoyer un dataset, je commence par 
    df.dropna() pour les valeurs manquantes, puis df.drop_duplicates(). J'utilise aussi df.describe() pour les statistiques 
    et df.groupby() pour les agrégations. Pour l'optimisation, je préfère les opérations vectorisées aux boucles.""",
    
    6: """Pour une API REST, je structurerais avec FastAPI : des endpoints clairs (/users, /posts), 
    validation Pydantic, gestion d'erreurs avec HTTPException, et documentation auto Swagger. 
    J'ajouterais aussi du rate limiting et de l'authentification JWT."""
}

print(" 4 réponses simulées prêtes pour évaluation")


 SIMULATION DE RÉPONSES
 4 réponses simulées prêtes pour évaluation


In [7]:
print("\n" + "="*60)
print(" ÉVALUATION DES RÉPONSES")
print("="*60)

evaluations = []

test_questions = questions['rh_questions'][:2] + questions['technical_questions'][:2]

for question in test_questions:
    q_id = question['id']
    
    if q_id not in simulated_answers:
        continue
    
    print(f"\n{'='*60}")
    print(f" Question {q_id}: {question['question'][:80]}...")
    print(f"\n Réponse candidat:")
    print(f"{simulated_answers[q_id][:150]}...")
    
    evaluation = simulator.evaluate_answer(
        question=question['question'],
        answer=simulated_answers[q_id],
        question_type=question['type'],
        target_skill=question.get('skill')
    )
    
    evaluations.append(evaluation)
    
    print(f"\n RÉSULTAT:")
    print(f"   Score: {evaluation['score']:.0f}/100")
    print(f"   Évaluation: {evaluation['evaluation'][:200]}...")
    
    print(f"\n Points forts:")
    for point in evaluation['points_forts'][:3]:
        print(f"   • {point}")
    
    print(f"\n  À améliorer:")
    for point in evaluation['points_amelioration'][:3]:
        print(f"   • {point}")


 ÉVALUATION DES RÉPONSES

 Question 1: Qu'est-ce qui vous motive à rejoindre notre équipe en tant que Junior ML Enginee...

 Réponse candidat:
Je suis diplômé d'un Master en IA et j'ai développé plusieurs projets en Machine Learning pendant mes stages. 
    J'ai notamment travaillé sur un sys...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 Réponse évaluée : 80/100

 RÉSULTAT:
   Score: 80/100
   Évaluation: Réponse claire et structurée, démontrant une bonne compréhension de l'apprentissage automatique et une motivation pour rejoindre l'équipe en tant que Junior ML Engineer....

 Points forts:
   • Exemples concrets mentionnés, tels que le système de recommandation avec TensorFlow
   • Bonne structure de réponse, facile à suivre
   • Démontre une passion pour l'apprentissage automatique et une volonté d'approfondir ses compétences

  À améliorer:
   • Pourrait ajouter plus de détails sur les défis rencontrés et sur la façon dont il les a résolus
   • Manque de mention de l'équipe ou de l'entreprise spécifiquement, et de ce que le candidat attend de son rôle

 Question 2: Parlez-moi de votre parcours académique et comment il vous a préparé pour ce pos...

 Réponse candidat:
Je souhaite rejoindre votre entreprise car vous êtes leaders dans l'IA. J'ai suivi vos publications techniques 
    et je suis impressionné par vos in

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 Réponse évaluée : 40/100

 RÉSULTAT:
   Score: 40/100
   Évaluation: Réponse qui montre de l'intérêt pour l'entreprise mais manque de clarté et de détails sur le parcours académique et les compétences techniques...

 Points forts:
   • Intérêt pour l'entreprise et ses innovations
   • Motivation pour apprendre et contribuer

  À améliorer:
   • Manque de détails sur le parcours académique
   • Absence de liens entre les études et le poste
   • Pas d'exemples concrets ou de compétences techniques mentionnées

 Question 5: Comment utilisez-vous les bibliothèques Python comme NumPy et Pandas pour l'anal...

 Réponse candidat:
En Python, j'utilise pandas pour la manipulation de données. Pour nettoyer un dataset, je commence par 
    df.dropna() pour les valeurs manquantes, p...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 Réponse évaluée : 80/100

 RÉSULTAT:
   Score: 80/100
   Évaluation: Réponse claire et structurée. Le candidat démontre une bonne compréhension des bibliothèques Python comme Pandas pour l'analyse de données....

 Points forts:
   • Exemples concrets mentionnés (df.dropna(), df.drop_duplicates(), df.describe(), df.groupby())
   • Bonne structure de réponse
   • Démontre l'autonomie dans l'utilisation des bibliothèques Python

  À améliorer:
   • Pourrait ajouter des métriques chiffrées pour étayer les exemples
   • Manque de détails techniques sur l'utilisation de NumPy

 Question 6: Expliquez les principes de base du Machine Learning et comment vous les avez app...

 Réponse candidat:
Pour une API REST, je structurerais avec FastAPI : des endpoints clairs (/users, /posts), 
    validation Pydantic, gestion d'erreurs avec HTTPExcepti...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 Réponse évaluée : 20/100

 RÉSULTAT:
   Score: 20/100
   Évaluation: La réponse ne répond pas à la question posée, elle parle d'API REST et de FastAPI au lieu de Machine Learning....

 Points forts:
   • Le candidat a une certaine connaissance des technologies liées au développement web

  À améliorer:
   • La réponse ne traite pas du sujet de la question
   • Manque complètement de contenu lié au Machine Learning
   • Le candidat ne démontre pas de compréhension des principes de base du Machine Learning


In [8]:
print("\n" + "="*60)
print(" FEEDBACK GLOBAL")
print("="*60)

global_feedback = simulator.generate_final_feedback(
    evaluations=evaluations,
    job_title=job['title']
)

print(f"\n Score global: {global_feedback['global_score']:.1f}/100")
print(f" Décision: {global_feedback['decision']}")

print(f"\n Synthèse:")
print(f"{global_feedback['synthese']}")

print(f"\n Compétences validées:")
for comp in global_feedback.get('competences_validees', [])[:5]:
    print(f"   • {comp}")

print(f"\n Axes de progression:")
for axe in global_feedback.get('axes_progression', [])[:5]:
    print(f"   • {axe}")

print(f"\n Prochaines étapes:")
for etape in global_feedback.get('prochaines_etapes', [])[:5]:
    print(f"   • {etape}")


 FEEDBACK GLOBAL


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


 Feedback global généré

 Score global: 55.0/100
 Décision: Prometteur

 Synthèse:
Le candidat a montré une passion pour l'apprentissage automatique et une bonne compréhension de certains concepts techniques, mais nécessite une amélioration sur les détails techniques et la liaison avec le rôle et l'entreprise. Les exemples concrets mentionnés, tels que le système de recommandation avec TensorFlow, sont un point fort. Cependant, il y a des domaines où le candidat doit approfondir ses connaissances et compétences pour correspondre pleinement aux exigences du poste de Junior ML Engineer.

 Compétences validées:
   • Connaissance de base de l'apprentissage automatique
   • Capacité à citer des exemples concrets d'applications de l'apprentissage automatique
   • Structure de réponse claire et facile à suivre
   • Motivation pour apprendre et contribuer à l'entreprise

 Axes de progression:
   • Développer des compétences techniques plus approfondies en Machine Learning
   • Améliorer la cap

In [9]:
# %% [markdown]
# ##  Résumé
# 
# - **Groq (Llama 3.3 70B)** : Génération de questions ultra-rapide
# -  **Questions personnalisées** : Basées sur CV + offre
# -  **Évaluation automatique** : Scoring 0-100 avec feedback
# -  **Feedback global** : Synthèse + recommandations
# 
# **Prochaine étape** : Intégrer dans l'API et Streamlit